# 00 — Pipeline consolidado de análisis comercial

Este notebook reemplaza la ejecución dispersa en múltiples notebooks y define un **pipeline único, trazable y mantenible**.

## Qué resuelve
- Un solo `BASE_DIR` (`C:\Python\trade`) para toda la operación.
- Sin rutas internas hardcodeadas: todo sale de una configuración central.
- Estructura robusta de carpetas para `downloads`, `inputs`, `outputs`, `logs` y `checkpoints`.
- Barra de progreso (`tqdm`) y logging completo en archivo + consola.
- Nombres de archivos de salida explícitos y autoexplicativos.

## Alcance analítico cubierto
1. Backbone geográfico (centroides + matriz OD).
2. Validación del panel comercial.
3. Barycenters esféricos (export/import).
4. Clustering gravitacional de trayectorias.
5. Moran's I agregado por año.

## Correcciones de fondo aplicadas

1. **Barycenter en esfera (corrección metodológica):**
   se eliminó el promedio directo de lat/lon y se implementó promedio ponderado en vectores 3D unitarios + renormalización.

2. **Moran con pesos robustos:**
   diagonal en cero, manejo explícito de distancias cero off-diagonal, y estandarización por fila segura.

3. **Validación de schema y tipos al cargar parquet:**
   control centralizado de columnas mínimas para evitar errores silenciosos entre etapas.

4. **Trazabilidad operativa real:**
   cada etapa escribe outputs con nombres claros y deja rastro en log timestamped.

In [ ]:
from __future__ import annotations

import logging
import math
import sys
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

In [ ]:
@dataclass
class PipelineConfig:
    base_dir: Path
    raw_trade_dir: Path
    geo_dir: Path
    downloads_dir: Path
    inputs_dir: Path
    outputs_dir: Path
    logs_dir: Path
    checkpoints_dir: Path
    stage_dir: dict[str, Path] = field(default_factory=dict)


def build_config(base_dir: str | Path, raw_trade_dir: str | Path | None = None) -> PipelineConfig:
    base = Path(base_dir)
    raw_trade = Path(raw_trade_dir) if raw_trade_dir else base / "dataverse_files"

    downloads = base / "downloads"
    inputs = base / "inputs"
    outputs = base / "outputs"
    logs = base / "logs"
    checkpoints = base / "checkpoints"
    geo = inputs / "geo"

    stage_dir = {
        "01_geo_backbone": outputs / "01_geo_backbone",
        "02_data_validation": outputs / "02_data_validation",
        "03_barycenters": outputs / "03_barycenters",
        "04_gravitational_clustering": outputs / "04_gravitational_clustering",
        "05_moran": outputs / "05_moran",
    }

    cfg = PipelineConfig(
        base_dir=base,
        raw_trade_dir=raw_trade,
        geo_dir=geo,
        downloads_dir=downloads,
        inputs_dir=inputs,
        outputs_dir=outputs,
        logs_dir=logs,
        checkpoints_dir=checkpoints,
        stage_dir=stage_dir,
    )
    return cfg


def ensure_directories(cfg: PipelineConfig) -> None:
    required = [
        cfg.base_dir,
        cfg.raw_trade_dir,
        cfg.geo_dir,
        cfg.downloads_dir,
        cfg.inputs_dir,
        cfg.outputs_dir,
        cfg.logs_dir,
        cfg.checkpoints_dir,
        *cfg.stage_dir.values(),
    ]
    for d in required:
        d.mkdir(parents=True, exist_ok=True)


# ===== Punto único de alta =====
BASE_DIR = Path(r"C:\Python\trade")
RAW_TRADE_DIR = Path(r"C:\Python\trade\dataverse_files")

CFG = build_config(BASE_DIR, RAW_TRADE_DIR)
ensure_directories(CFG)
CFG

In [ ]:
def setup_logger(log_dir: Path, logger_name: str = "trade_pipeline") -> tuple[logging.Logger, Path]:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = log_dir / f"trade_pipeline_{ts}.log"

    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    file_handler = logging.FileHandler(log_file, encoding="utf-8")
    file_handler.setFormatter(formatter)
    file_handler.setLevel(logging.INFO)

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(formatter)
    stream_handler.setLevel(logging.INFO)

    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)
    logger.propagate = False

    return logger, log_file


LOGGER, LOG_FILE = setup_logger(CFG.logs_dir)
LOGGER.info("Pipeline inicializado")
LOGGER.info("BASE_DIR=%s", CFG.base_dir)
LOGGER.info("RAW_TRADE_DIR=%s", CFG.raw_trade_dir)
print(f"Log activo: {LOG_FILE}")

In [ ]:
REQUIRED_TRADE_COLUMNS = {"year", "location_code", "partner_code", "value_final"}


def list_trade_files(raw_trade_dir: Path) -> list[Path]:
    files = sorted(raw_trade_dir.glob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"No se encontraron archivos parquet en {raw_trade_dir}")
    return files


def validate_trade_schema(df: pd.DataFrame, file_name: str) -> None:
    missing = REQUIRED_TRADE_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"{file_name}: faltan columnas requeridas: {sorted(missing)}")


def load_trade_panel(cfg: PipelineConfig, max_files: int | None = None) -> pd.DataFrame:
    files = list_trade_files(cfg.raw_trade_dir)
    if max_files is not None:
        files = files[:max_files]

    chunks = []
    for fp in tqdm(files, desc="Cargando parquet de comercio"):
        df = pd.read_parquet(fp)
        validate_trade_schema(df, fp.name)
        slim = df[["year", "location_code", "partner_code", "value_final"]].copy()
        chunks.append(slim)

    panel = pd.concat(chunks, ignore_index=True)
    panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype("Int64")
    panel["value_final"] = pd.to_numeric(panel["value_final"], errors="coerce").fillna(0.0)
    panel["location_code"] = panel["location_code"].astype(str)
    panel["partner_code"] = panel["partner_code"].astype(str)

    LOGGER.info("Panel consolidado: %s filas, %s columnas", panel.shape[0], panel.shape[1])
    return panel

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * R * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))


def latlon_to_unitvec(lat_deg: np.ndarray, lon_deg: np.ndarray) -> np.ndarray:
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    x = np.cos(lat) * np.cos(lon)
    y = np.cos(lat) * np.sin(lon)
    z = np.sin(lat)
    return np.column_stack([x, y, z])


def unitvec_to_latlon(v: np.ndarray) -> tuple[float, float]:
    x, y, z = v
    lon = math.degrees(math.atan2(y, x))
    hyp = math.sqrt(x * x + y * y)
    lat = math.degrees(math.atan2(z, hyp))
    return lat, lon


def spherical_weighted_barycenter(lat: pd.Series, lon: pd.Series, w: pd.Series) -> tuple[float, float]:
    weights = pd.to_numeric(w, errors="coerce").fillna(0.0).to_numpy(float)
    if weights.sum() <= 0:
        return np.nan, np.nan

    vec = latlon_to_unitvec(lat.to_numpy(float), lon.to_numpy(float))
    weighted = (vec * weights[:, None]).sum(axis=0)
    norm = np.linalg.norm(weighted)
    if norm == 0:
        return np.nan, np.nan

    return unitvec_to_latlon(weighted / norm)

In [ ]:
# ---------- ETAPA 01 ----------
def run_stage_01_geo_backbone(cfg: PipelineConfig) -> dict[str, Path]:
    """
    Espera archivo base de centroides en:
    inputs/geo/country_centroids_augmented.csv
    con columnas: code, lat, lon
    """
    out_dir = cfg.stage_dir["01_geo_backbone"]
    out_dir.mkdir(parents=True, exist_ok=True)

    centroids_file = cfg.geo_dir / "country_centroids_augmented.csv"
    if not centroids_file.exists():
        raise FileNotFoundError(
            f"Falta {centroids_file}. Copiar/descargar centroides y reintentar."
        )

    cent = pd.read_csv(centroids_file)
    cent = cent[["code", "lat", "lon"]].dropna().copy()
    cent["code"] = cent["code"].astype(str)

    # Matriz OD completa
    codes = cent["code"].tolist()
    lat = cent["lat"].to_numpy(float)
    lon = cent["lon"].to_numpy(float)

    D = np.zeros((len(codes), len(codes)), dtype=float)
    for i in tqdm(range(len(codes)), desc="Calculando matriz OD (km)"):
        D[i, :] = haversine_km(lat[i], lon[i], lat, lon)

    od_df = pd.DataFrame(D, index=codes, columns=codes)

    cent_out = out_dir / "country_centroids_clean.csv"
    od_out = out_dir / "od_distance_matrix_km.csv"

    cent.to_csv(cent_out, index=False)
    od_df.to_csv(od_out)

    LOGGER.info("Etapa 01 finalizada | centroides=%s | od=%s", cent_out, od_out)
    return {"centroids": cent_out, "od_matrix": od_out}

In [ ]:
# ---------- ETAPA 02 ----------
def run_stage_02_data_validation(panel: pd.DataFrame, cfg: PipelineConfig) -> dict[str, Path]:
    out_dir = cfg.stage_dir["02_data_validation"]
    out_dir.mkdir(parents=True, exist_ok=True)

    exp = panel.groupby(["year", "location_code"], as_index=False)["value_final"].sum()
    exp = exp.rename(columns={"location_code": "code", "value_final": "exports_total"})

    imp = panel.groupby(["year", "partner_code"], as_index=False)["value_final"].sum()
    imp = imp.rename(columns={"partner_code": "code", "value_final": "imports_total"})

    audit = exp.merge(imp, on=["year", "code"], how="outer").fillna(0.0)
    audit["trade_balance_proxy"] = audit["exports_total"] - audit["imports_total"]

    coverage = panel.groupby("year", as_index=False).agg(
        reporters=("location_code", "nunique"),
        partners=("partner_code", "nunique"),
        total_trade_value=("value_final", "sum"),
    )

    audit_out = out_dir / "validation_country_year_exports_imports.csv"
    coverage_out = out_dir / "validation_yearly_coverage_and_totals.csv"

    audit.to_csv(audit_out, index=False)
    coverage.to_csv(coverage_out, index=False)

    LOGGER.info("Etapa 02 finalizada | audit=%s | coverage=%s", audit_out, coverage_out)
    return {"audit": audit_out, "coverage": coverage_out}

In [ ]:
# ---------- ETAPA 03 ----------
def run_stage_03_barycenters(panel: pd.DataFrame, cfg: PipelineConfig, centroids_file: Path | None = None) -> Path:
    out_dir = cfg.stage_dir["03_barycenters"]
    out_dir.mkdir(parents=True, exist_ok=True)

    centroids_path = centroids_file or (cfg.stage_dir["01_geo_backbone"] / "country_centroids_clean.csv")
    if not centroids_path.exists():
        centroids_path = cfg.geo_dir / "country_centroids_augmented.csv"
    if not centroids_path.exists():
        raise FileNotFoundError("No hay centroides disponibles para etapa 03.")

    cent = pd.read_csv(centroids_path)[["code", "lat", "lon"]].dropna().copy()
    cent["code"] = cent["code"].astype(str)

    keys = panel[["year", "location_code"]].drop_duplicates().sort_values(["year", "location_code"])

    records = []
    for row in tqdm(keys.itertuples(index=False), total=len(keys), desc="Barycenters país-año"):
        year = int(row.year)
        reporter = str(row.location_code)

        exports_edges = panel[(panel["year"] == year) & (panel["location_code"] == reporter)].copy()
        imports_edges = panel[(panel["year"] == year) & (panel["partner_code"] == reporter)].copy()

        exp_geo = exports_edges.merge(cent, left_on="partner_code", right_on="code", how="left").dropna(subset=["lat", "lon"])
        imp_geo = imports_edges.merge(cent, left_on="location_code", right_on="code", how="left").dropna(subset=["lat", "lon"])

        lat_e, lon_e = spherical_weighted_barycenter(exp_geo["lat"], exp_geo["lon"], exp_geo["value_final"]) if not exp_geo.empty else (np.nan, np.nan)
        lat_i, lon_i = spherical_weighted_barycenter(imp_geo["lat"], imp_geo["lon"], imp_geo["value_final"]) if not imp_geo.empty else (np.nan, np.nan)

        records.append(
            {
                "year": year,
                "code": reporter,
                "lat_exports": lat_e,
                "lon_exports": lon_e,
                "lat_imports": lat_i,
                "lon_imports": lon_i,
                "exports_total_value_final": float(exports_edges["value_final"].sum()),
                "imports_total_value_final": float(imports_edges["value_final"].sum()),
            }
        )

    out = pd.DataFrame(records)
    out_file = out_dir / "barycenters_country_year_spherical.csv"
    out.to_csv(out_file, index=False)

    LOGGER.info("Etapa 03 finalizada | barycenters=%s", out_file)
    return out_file

In [ ]:
# ---------- ETAPA 04 ----------
def build_trajectory_distance_matrix(bary_df: pd.DataFrame, lat_col: str, lon_col: str) -> tuple[np.ndarray, list[str]]:
    bary_df = bary_df.dropna(subset=[lat_col, lon_col, "year", "code"]).copy()
    countries = sorted(bary_df["code"].astype(str).unique())

    series = {}
    for c in countries:
        sub = bary_df[bary_df["code"] == c].sort_values("year")
        series[c] = sub[["year", lat_col, lon_col]].set_index("year")

    n = len(countries)
    D = np.zeros((n, n), dtype=float)

    for i in tqdm(range(n), desc=f"Distancias de trayectorias ({lat_col})"):
        ci = countries[i]
        for j in range(i + 1, n):
            cj = countries[j]
            common = series[ci].join(series[cj], how="inner", lsuffix="_i", rsuffix="_j")
            if common.empty:
                dist = np.nan
            else:
                d = haversine_km(
                    common[f"{lat_col}_i"].to_numpy(float),
                    common[f"{lon_col}_i"].to_numpy(float),
                    common[f"{lat_col}_j"].to_numpy(float),
                    common[f"{lon_col}_j"].to_numpy(float),
                )
                dist = float(np.nanmean(d))
            D[i, j] = dist
            D[j, i] = dist

    np.fill_diagonal(D, 0.0)

    # impute faltantes con mediana de off-diagonal válidos
    tri = D[np.triu_indices(n, k=1)]
    finite = tri[np.isfinite(tri)]
    fallback = float(np.nanmedian(finite)) if len(finite) else 0.0
    D[~np.isfinite(D)] = fallback

    return D, countries


def run_stage_04_gravitational_clustering(cfg: PipelineConfig, barycenters_file: Path, n_clusters: int = 4) -> dict[str, Path]:
    out_dir = cfg.stage_dir["04_gravitational_clustering"]
    out_dir.mkdir(parents=True, exist_ok=True)

    b = pd.read_csv(barycenters_file)

    D, countries = build_trajectory_distance_matrix(b, "lat_exports", "lon_exports")
    condensed = squareform(D, checks=False)
    Z = linkage(condensed, method="average")
    labels = fcluster(Z, t=n_clusters, criterion="maxclust")

    clusters = pd.DataFrame({"code": countries, "cluster_exports": labels})

    dist_out = out_dir / "trajectory_distance_matrix_exports_km.csv"
    cluster_out = out_dir / f"clusters_exports_k{n_clusters}.csv"

    pd.DataFrame(D, index=countries, columns=countries).to_csv(dist_out)
    clusters.to_csv(cluster_out, index=False)

    LOGGER.info("Etapa 04 finalizada | dist=%s | clusters=%s", dist_out, cluster_out)
    return {"distance_matrix": dist_out, "clusters": cluster_out}

In [ ]:
# ---------- ETAPA 05 ----------
def inverse_distance_weights(lat: np.ndarray, lon: np.ndarray) -> np.ndarray:
    n = len(lat)
    D = np.zeros((n, n), dtype=float)
    for i in range(n):
        D[i, :] = haversine_km(lat[i], lon[i], lat, lon)

    with np.errstate(divide="ignore", invalid="ignore"):
        W = 1.0 / D

    np.fill_diagonal(W, 0.0)
    W[~np.isfinite(W)] = 0.0

    row_sum = W.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    W = W / row_sum
    return W


def moran_i(x: np.ndarray, W: np.ndarray) -> float:
    z = x - x.mean()
    den = float(z @ z)
    if den == 0:
        return np.nan
    s0 = float(W.sum())
    if s0 == 0:
        return np.nan
    num = float(z @ W @ z)
    n = len(z)
    return (n / s0) * (num / den)


def run_stage_05_moran_aggregate(panel: pd.DataFrame, cfg: PipelineConfig, centroids_file: Path | None = None) -> Path:
    out_dir = cfg.stage_dir["05_moran"]
    out_dir.mkdir(parents=True, exist_ok=True)

    centroids_path = centroids_file or (cfg.stage_dir["01_geo_backbone"] / "country_centroids_clean.csv")
    if not centroids_path.exists():
        centroids_path = cfg.geo_dir / "country_centroids_augmented.csv"
    if not centroids_path.exists():
        raise FileNotFoundError("No hay centroides disponibles para etapa 05.")

    c = pd.read_csv(centroids_path)[["code", "lat", "lon"]].dropna().copy()
    c["code"] = c["code"].astype(str)

    results = []
    for yr in tqdm(sorted(panel["year"].dropna().astype(int).unique()), desc="Moran I agregado por año"):
        ydf = panel[panel["year"] == yr]

        exp = ydf.groupby("location_code", as_index=False)["value_final"].sum().rename(columns={"location_code": "code", "value_final": "exports_total"})
        data = c.merge(exp, on="code", how="left").fillna({"exports_total": 0.0})

        lat = data["lat"].to_numpy(float)
        lon = data["lon"].to_numpy(float)
        x = data["exports_total"].to_numpy(float)

        W = inverse_distance_weights(lat, lon)
        I = moran_i(x, W)

        results.append({
            "year": int(yr),
            "moran_i_exports_aggregate": I,
            "n_countries": int(len(data)),
            "total_exports_value_final": float(x.sum()),
        })

    out = pd.DataFrame(results)
    out_file = out_dir / "moran_i_exports_aggregate_by_year.csv"
    out.to_csv(out_file, index=False)

    LOGGER.info("Etapa 05 finalizada | moran=%s", out_file)
    return out_file

In [ ]:
# =========================
# Runner principal
# =========================
# Para prueba rápida: usar un entero (p.ej. 2). Para corrida real: None.
MAX_FILES_FOR_SMOKE = 2
RUN_STAGE_01 = True
RUN_STAGE_02 = True
RUN_STAGE_03 = True
RUN_STAGE_04 = True
RUN_STAGE_05 = True
N_CLUSTERS_STAGE_04 = 4

LOGGER.info("Iniciando runner principal")

panel = load_trade_panel(CFG, max_files=MAX_FILES_FOR_SMOKE)

if RUN_STAGE_01:
    stage01 = run_stage_01_geo_backbone(CFG)
else:
    stage01 = {}

if RUN_STAGE_02:
    stage02 = run_stage_02_data_validation(panel, CFG)

if RUN_STAGE_03:
    bary_file = run_stage_03_barycenters(panel, CFG, centroids_file=stage01.get("centroids"))
else:
    bary_file = CFG.stage_dir["03_barycenters"] / "barycenters_country_year_spherical.csv"

if RUN_STAGE_04 and bary_file.exists():
    stage04 = run_stage_04_gravitational_clustering(CFG, bary_file, n_clusters=N_CLUSTERS_STAGE_04)

if RUN_STAGE_05:
    moran_file = run_stage_05_moran_aggregate(panel, CFG, centroids_file=stage01.get("centroids"))

LOGGER.info("Pipeline finalizado sin errores fatales")
print(f"Revisar log en: {LOG_FILE}")

## Operación recomendada en producción

1. Dejar `MAX_FILES_FOR_SMOKE = None`.
2. Ejecutar todas las celdas en orden.
3. Validar outputs en `outputs/01...05`.
4. Si falla, compartir el log (`logs/trade_pipeline_YYYYMMDD_HHMMSS.log`) para diagnóstico.

## Estructura final esperada
- `C:\Python\trade\dataverse_files` → parquet originales (input canónico)
- `C:\Python\trade\inputs\geo` → insumos geográficos
- `C:\Python\trade\outputs\01_geo_backbone`
- `C:\Python\trade\outputs\02_data_validation`
- `C:\Python\trade\outputs\03_barycenters`
- `C:\Python\trade\outputs\04_gravitational_clustering`
- `C:\Python\trade\outputs\05_moran`
- `C:\Python\trade\logs`
- `C:\Python\trade\checkpoints`